Fetches and parses settled historical market data from Kalshi's API.
Docs:     https://docs.kalshi.com/api-reference/historical/get-historical-markets

In [ ]:
import requests
import json
import time
from datetime import datetime, timezone

from config.settings import (
    KALSHI_HIST_MARKETS_URL,
    KALSHI_START_DATE,
    FETCH_LIMIT,
    REQUEST_TIMEOUT_SECONDS,
    PAGINATION_DELAY_SECONDS,
)

In [ ]:
def fetch_historical_markets(limit: int = FETCH_LIMIT) -> list[dict]:
    """
    Paginate through all Kalshi historical markets, stopping once markets
    become older than KALSHI_START_DATE.

    Pagination:
      The response includes a `cursor` string. Pass it as ?cursor=X in the
      next request to get the next page. An empty cursor means no more pages.

    Performance note:
      At FETCH_LIMIT=1000, each page covers ~1000 markets. For a Jan 2025
      start date you may need 50–200 pages depending on sports market volume.
      PAGINATION_DELAY_SECONDS adds a small pause between pages to be polite.
    """
    cutoff    = datetime.fromisoformat(KALSHI_START_DATE.replace("Z", "+00:00"))
    all_data  = []
    cursor    = ""
    page      = 1

    while True:
        params = {"limit": limit}
        if cursor:
            params["cursor"] = cursor

        print(f"[Kalshi] Fetching page {page} ({limit} markets per page)...")
        response = requests.get(KALSHI_HIST_MARKETS_URL, params=params, timeout=REQUEST_TIMEOUT_SECONDS)
        response.raise_for_status()

        data   = response.json()
        batch  = data.get("markets", [])
        cursor = data.get("cursor", "")

        if not batch:
            break

        all_data.extend(batch)
        print(f"[Kalshi] Page {page}: {len(batch)} markets | total so far: {len(all_data)}")

        # Early stop: the endpoint returns newest-first. Once the last market
        # in a page is older than our cutoff, we have everything we need.
        oldest_close = batch[-1].get("close_time", "")
        if oldest_close:
            oldest_dt = datetime.fromisoformat(oldest_close.replace("Z", "+00:00"))
            if oldest_dt < cutoff:
                print(f"[Kalshi] Reached markets older than {KALSHI_START_DATE} — stopping.")
                break

        if not cursor:
            print("[Kalshi] No more pages.")
            break

        page += 1
        time.sleep(PAGINATION_DELAY_SECONDS)

    print(f"[Kalshi] Total fetched before date filter: {len(all_data)}")
    return all_data

In [ ]:
def parse_historical_markets(raw_markets: list[dict], start_date: str = KALSHI_START_DATE) -> list[dict]:
    """
    Extract the requested fields from raw historical market records,
    filtering out markets with close_time before KALSHI_START_DATE.

    Field notes:
      ticker               — unique market ID, e.g. "FED-26MAY-N0.25"
      event_ticker         — parent event, e.g. "FED-26MAY"
      market_type          — "binary" or "scalar"
      title / subtitle     — human-readable question and context
      open_time            — when trading opened (ISO datetime)
      close_time           — when trading closed (ISO datetime)
      expiration_time      — when outcome was officially determined (ISO datetime)
      status               — always "settled" from this endpoint
      yes_bid_dollars      — last best bid for YES before settlement (dollar string)
      yes_bid_size_fp      — contracts available at that bid (fixed-point string)
      no_bid_dollars       — last best bid for NO before settlement
      volume_fp            — total contracts traded, e.g. "48200.50"
      result               — "yes" or "no" — THE FINAL OUTCOME.
                             Only exists on the historical endpoint.
      liquidity_dollars    — liquidity at time of settlement snapshot
      price_ranges         — for scalar markets: numeric bands traders bet on,
                             stored as a JSON string in the CSV.
                             e.g. '[{"start":"3.0","end":"3.5","step":"0.1"}]'
                             Binary markets → "[]"
      settlement_value_dollars — real-world value that triggered resolution.
                             For scalar markets: e.g. "3.20" for a CPI market.
                             For binary: "1.00" (YES wins) or "0.00" (NO wins).
      settlement_ts        — exact timestamp when payouts were processed.
    """
    cutoff = datetime.fromisoformat(start_date.replace("Z", "+00:00"))
    rows   = []

    for m in raw_markets:

        # Date filter — discard markets before the cutoff
        close_time_str = m.get("close_time", "")
        if close_time_str:
            close_dt = datetime.fromisoformat(close_time_str.replace("Z", "+00:00"))
            if close_dt < cutoff:
                continue

        # price_ranges: store as JSON string so it fits in one CSV cell.
        # In Excel: looks like '[{"start":"3.0","end":"4.0","step":"0.1"}]'
        price_ranges_str = json.dumps(m.get("price_ranges") or [])

        rows.append({
            "ticker":                   m.get("ticker"),
            "event_ticker":             m.get("event_ticker"),
            "market_type":              m.get("market_type"),
            "title":                    m.get("title"),
            "subtitle":                 m.get("subtitle"),
            "open_time":                m.get("open_time"),
            "close_time":               close_time_str,
            "expiration_time":          m.get("expiration_time"),
            "status":                   m.get("status"),
            "yes_bid_dollars":          m.get("yes_bid_dollars"),
            "yes_bid_size_fp":          m.get("yes_bid_size_fp"),
            "no_bid_dollars":           m.get("no_bid_dollars"),
            "volume_fp":                m.get("volume_fp"),
            "result":                   m.get("result"),
            "liquidity_dollars":        m.get("liquidity_dollars"),
            "price_ranges":             price_ranges_str,
            "settlement_value_dollars": m.get("settlement_value_dollars"),
            "settlement_ts":            m.get("settlement_ts"),
        })

    return rows